In [1]:
import pandas as pd
import numpy as np

In [2]:

# Instellingen om te tweaken
# Aantal aanbevelingen; bepaalt ook de K in Precision@K en NDCG@K.
TOP_K = 3
# Maximumaantal meest vergelijkbare studenten dat voor voorspellingen wordt gebruikt.
N_NEIGHBORS = 5
# Minimumaantal gedeelde materialen waarop een studentcorrelatie wordt berekend.
MIN_COMMON_ITEMS = 3
# Minimumaantal gekozen buren dat een materiaal moet hebben beoordeeld.
MIN_ITEM_SUPPORT = 1
# Vanaf deze intscore telt een achtergehouden materiaal als relevant.
RELEVANCE_THRESHOLD = 4
# Deel van de interactieparen per student dat als testdata wordt achtergehouden.
TEST_FRACTION = 0.3
# Verschillende seeds voor herhaalde, reproduceerbare train/test-splits.
# Gebruik dezelfde lijst wanneer je verschillende instellingen vergelijkt.
RANDOM_STATES = [11, 21, 31, 41, 51, 61, 71, 81, 91, 1]
# True gebruikt alleen positief gecorreleerde buren; False laat ook andere correlaties toe.
ONLY_POSITIVE_NEIGHBORS = True
# Kies componenten voor de interactiescore door regels toe te voegen of weg te laten.
# De gewichten geven het relatieve belang; ze worden automatisch genormaliseerd.
SCORE_WEIGHTS = {
    'leesprc': 0.15,      # Afgeleid van leespercentage (omgezet naar een 0-5-schaal).
    'inttype': 0.25,      # Afgeleid van interactietype via eval_type_scores hieronder.
    'beoordeling': 0.75,  # De beoordeling uit de interactiedata.
    'leestijd_min': 0.15,  # Leestijd lineair geschaald naar 0-5.
}

# Lees de opgeschoonde data en bereken de interactiescore, onafhankelijk van de baselinecellen.
eval_raw = pd.read_csv('./dataset/cleaned_Interacties.csv')

eval_type_scores = {
    'geliket': 5.0,
    'opgeslagen': 4.0,
    'gelezen': 3.0,
    'bekeken': 2.0,
    'niet_afgemaakt': 1.0,
    'afgewezen': 1.0,
}
eval_unknown_types = (
    set(eval_raw['interactietype'].dropna().unique()) - set(eval_type_scores)
)
if eval_unknown_types:
    raise ValueError('Onbekende interactietypes: {}'.format(sorted(eval_unknown_types)))

# Schaal leestijd lineair naar 0-5 zodat deze vergelijkbaar is met de andere componenten.
eval_reading_time = pd.to_numeric(eval_raw['leestijd_min'], errors='coerce')
eval_reading_time_min = eval_reading_time.min()
eval_reading_time_max = eval_reading_time.max()
if pd.isna(eval_reading_time_min) or pd.isna(eval_reading_time_max):
    raise ValueError('Geen geldige waarden gevonden voor leestijd_min.')
if eval_reading_time_max == eval_reading_time_min:
    raise ValueError('leestijd_min kan niet lineair worden geschaald als alle waarden gelijk zijn.')
eval_reading_time_score = (
    (eval_reading_time - eval_reading_time_min)
    / (eval_reading_time_max - eval_reading_time_min)
    * 5
)

eval_all_inputs = pd.DataFrame({
    'leesprc': (eval_raw['leespercentage'] / 20).round(),
    'inttype': eval_raw['interactietype'].map(eval_type_scores),
    'beoordeling': pd.to_numeric(eval_raw['beoordeling'], errors='coerce'),
    'leestijd_min': eval_reading_time_score,
})
if not SCORE_WEIGHTS:
    raise ValueError('SCORE_WEIGHTS moet minstens één scorecomponent bevatten.')
eval_unknown_components = set(SCORE_WEIGHTS) - set(eval_all_inputs.columns)
if eval_unknown_components:
    raise ValueError('Onbekende scorecomponenten: {}'.format(sorted(eval_unknown_components)))
if any(weight <= 0 for weight in SCORE_WEIGHTS.values()):
    raise ValueError('Alle gekozen scoregewichten moeten groter zijn dan 0.')
eval_inputs = eval_all_inputs[list(SCORE_WEIGHTS)]
eval_weights = pd.Series(SCORE_WEIGHTS)
eval_weighted_sum = eval_inputs.mul(eval_weights, axis='columns').sum(axis=1, min_count=1)
eval_available_weight = eval_inputs.notna().mul(eval_weights, axis='columns').sum(axis=1)
eval_raw['intscore'] = eval_weighted_sum / eval_available_weight

# Gemiddelde score per student-materiaalpaar; voorkom overlap tussen train en test.
eval_pairs = (
    eval_raw.groupby(['student_id', 'cataloguscode'], as_index=False)['intscore']
    .mean()
    .dropna(subset=['intscore'])
    .reset_index(drop=True)
)

# Voer voor elke seed de volledige split, aanbevelingen en evaluatie opnieuw uit.
def evaluate_one_split(eval_random_state):
    eval_rng = np.random.default_rng(eval_random_state)
    eval_test_indices = []
    for _, eval_user_pairs in eval_pairs.groupby('student_id', sort=False):
        if len(eval_user_pairs) < 2:
            continue
        eval_n_test = max(1, round(len(eval_user_pairs) * TEST_FRACTION))
        eval_n_test = min(eval_n_test, len(eval_user_pairs) - 1)
        eval_seed = int(eval_rng.integers(0, 2**32 - 1))
        eval_test_indices.extend(
            eval_user_pairs.sample(n=eval_n_test, random_state=eval_seed).index
        )

    if not eval_test_indices:
        raise ValueError('Niet genoeg student-materiaalparen voor de train/test-split.')

    eval_test = eval_pairs.loc[eval_test_indices].copy()
    eval_train = eval_pairs.drop(index=eval_test_indices).copy()
    eval_students = eval_pairs['student_id'].unique()
    eval_items = eval_pairs['cataloguscode'].unique()
    eval_train_matrix = (
        eval_train.pivot(index='student_id', columns='cataloguscode', values='intscore')
        .reindex(index=eval_students, columns=eval_items)
    )
    eval_similarity = eval_train_matrix.T.corr(min_periods=MIN_COMMON_ITEMS)

    def predict_scores_for_split(user_id):
        eval_neighbors = eval_similarity.loc[user_id].drop(index=user_id).dropna()
        if ONLY_POSITIVE_NEIGHBORS:
            eval_neighbors = eval_neighbors[eval_neighbors > 0]
        eval_neighbors = eval_neighbors.nlargest(N_NEIGHBORS)
        if eval_neighbors.empty:
            return pd.Series(dtype='float64')

        eval_neighbor_scores = eval_train_matrix.loc[eval_neighbors.index]
        eval_weighted_sum = eval_neighbor_scores.mul(
            eval_neighbors, axis=0
        ).sum(axis=0, min_count=1)
        eval_weight_sum = eval_neighbor_scores.notna().mul(
            eval_neighbors, axis=0
        ).sum(axis=0)
        eval_predicted = eval_weighted_sum / eval_weight_sum
        eval_support = eval_neighbor_scores.notna().sum(axis=0)
        eval_unseen = eval_train_matrix.loc[user_id].isna()
        eval_candidates = eval_unseen & (eval_support >= MIN_ITEM_SUPPORT)
        return eval_predicted[eval_candidates]

    eval_test_users = eval_test['student_id'].unique()
    eval_predicted_by_user = {
        eval_user_id: predict_scores_for_split(eval_user_id)
        for eval_user_id in eval_test_users
    }
    eval_recs_by_user = {
        eval_user_id: eval_scores.nlargest(TOP_K).index.tolist()
        for eval_user_id, eval_scores in eval_predicted_by_user.items()
    }
    eval_relevant_by_user = (
        eval_test.loc[eval_test['intscore'] >= RELEVANCE_THRESHOLD]
        .groupby('student_id')['cataloguscode']
        .apply(set)
        .to_dict()
    )
    eval_actual_scores = []
    eval_predicted_scores = []
    for eval_row in eval_test.itertuples(index=False):
        eval_prediction = eval_predicted_by_user[eval_row.student_id].get(
            eval_row.cataloguscode, np.nan
        )
        if pd.notna(eval_prediction):
            eval_actual_scores.append(eval_row.intscore)
            eval_predicted_scores.append(eval_prediction)
    if eval_predicted_scores:
        eval_errors = np.asarray(eval_actual_scores) - np.asarray(eval_predicted_scores)
        eval_mae = np.mean(np.abs(eval_errors))
        eval_rmse = np.sqrt(np.mean(np.square(eval_errors)))
        eval_train_mean = eval_train['intscore'].mean()
        eval_baseline_errors = np.asarray(eval_actual_scores) - eval_train_mean
        eval_baseline_mae = np.mean(np.abs(eval_baseline_errors))
        eval_baseline_rmse = np.sqrt(np.mean(np.square(eval_baseline_errors)))
    else:
        eval_mae = np.nan
        eval_rmse = np.nan
        eval_baseline_mae = np.nan
        eval_baseline_rmse = np.nan

    eval_precision_values = []
    eval_ndcg_values = []
    for eval_user_id, eval_relevant in eval_relevant_by_user.items():
        eval_recs = eval_recs_by_user[eval_user_id]
        eval_hits = [item in eval_relevant for item in eval_recs]
        eval_precision_values.append(sum(eval_hits) / TOP_K)
        eval_dcg = sum(
            1 / np.log2(rank + 1)
            for rank, is_relevant in enumerate(eval_hits, start=1)
            if is_relevant
        )
        eval_ideal_hits = min(TOP_K, len(eval_relevant))
        eval_idcg = sum(
            1 / np.log2(rank + 1)
            for rank in range(1, eval_ideal_hits + 1)
        )
        eval_ndcg_values.append(eval_dcg / eval_idcg if eval_idcg else 0.0)

    eval_known_items_by_user = (
        eval_test.groupby('student_id')['cataloguscode'].apply(set).to_dict()
    )
    eval_judged_precision_values = []
    eval_judged_shares = []
    eval_top_k_coverages = []
    eval_judged_user_count = 0
    for eval_user_id, eval_known_items in eval_known_items_by_user.items():
        eval_recs = eval_recs_by_user[eval_user_id]
        eval_judged_recs = [item for item in eval_recs if item in eval_known_items]
        eval_relevant = eval_relevant_by_user.get(eval_user_id, set())
        eval_judged_hits = sum(item in eval_relevant for item in eval_judged_recs)
        if eval_judged_recs:
            eval_judged_user_count += 1
            eval_judged_precision_values.append(
                eval_judged_hits / len(eval_judged_recs)
            )
        else:
            eval_judged_precision_values.append(np.nan)
        eval_judged_shares.append(
            len(eval_judged_recs) / len(eval_recs) if eval_recs else np.nan
        )
        eval_top_k_coverages.append(len(eval_judged_recs) / TOP_K)

    eval_total_pairs = len(eval_pairs)
    return {
        'random_state': eval_random_state,
        'train_percent': 100 * len(eval_train) / eval_total_pairs,
        'test_percent': 100 * len(eval_test) / eval_total_pairs,
        'students_with_relevant_test': len(eval_precision_values),
        'students_with_judged_recommendations': eval_judged_user_count,
        'average_recommendations_returned': pd.Series(
            [len(recs) for recs in eval_recs_by_user.values()], dtype='float64'
        ).mean(),
        'Precision@K': pd.Series(eval_precision_values, dtype='float64').mean(),
        'NDCG@K': pd.Series(eval_ndcg_values, dtype='float64').mean(),
        'Judged-only Precision@K': pd.Series(
            eval_judged_precision_values, dtype='float64'
        ).mean(),
        'judged_share_of_recommendations': pd.Series(
            eval_judged_shares, dtype='float64'
        ).mean(),
        'judged_top_k_coverage': pd.Series(
            eval_top_k_coverages, dtype='float64'
        ).mean(),
        'MAE': eval_mae,
        'RMSE': eval_rmse,
        'Global-mean baseline MAE': eval_baseline_mae,
        'Global-mean baseline RMSE': eval_baseline_rmse,
        'score_prediction_coverage': len(eval_predicted_scores) / len(eval_test),
        'predicted_test_pairs': len(eval_predicted_scores),
    }

if not RANDOM_STATES:
    raise ValueError('RANDOM_STATES moet minstens ??n seed bevatten.')
if len(set(RANDOM_STATES)) != len(RANDOM_STATES):
    raise ValueError('RANDOM_STATES mag geen dubbele seeds bevatten.')

eval_run_metrics = pd.DataFrame(
    [evaluate_one_split(eval_seed) for eval_seed in RANDOM_STATES]
)

eval_component_sources = {
    'leesprc': 'leespercentage / 20, afgerond',
    'inttype': 'interactietype (gescoord via eval_type_scores)',
    'beoordeling': 'beoordeling',
    'leestijd_min': 'leestijd_min lineair geschaald naar 0-5',
}
print('Gebruikte scorecomponenten en ingestelde relatieve gewichten:')
for eval_component, eval_weight in SCORE_WEIGHTS.items():
    print('  {} <- {}: {}'.format(
        eval_component, eval_component_sources[eval_component], eval_weight
    ))
print('Instellingen (1/2): splits={} | test={:.0f}% | train/test={:.1f}/{:.1f}% | K={}'.format(
    len(RANDOM_STATES), 100 * TEST_FRACTION,
    eval_run_metrics['train_percent'].mean(), eval_run_metrics['test_percent'].mean(),
    TOP_K
))
print('Instellingen (2/2): buren={} | overlap={} | support={} | drempel={} | seeds={}'.format(
    N_NEIGHBORS, MIN_COMMON_ITEMS, MIN_ITEM_SUPPORT, RELEVANCE_THRESHOLD, RANDOM_STATES
))
print('Gemiddelde +/- standaardafwijking tussen splits:')

def print_run_summary(eval_label, eval_column):
    eval_values = eval_run_metrics[eval_column].dropna()
    if eval_values.empty:
        print('{}: geen geldige resultaten'.format(eval_label))
        return
    print('{}: {:.4f} +/- {:.4f}'.format(
        eval_label, eval_values.mean(), eval_values.std(ddof=0)
    ))

for eval_label, eval_column in [
    ('Precision@{}'.format(TOP_K), 'Precision@K'),
    ('NDCG@{}'.format(TOP_K), 'NDCG@K'),
    ('Judged-only Precision@{}'.format(TOP_K), 'Judged-only Precision@K'),
    ('Aandeel beoordeeld in aanbevelingen', 'judged_share_of_recommendations'),
    ('Studenten met relevante testitems', 'students_with_relevant_test'),
    ('Studenten met beoordeelde aanbevelingen', 'students_with_judged_recommendations'),
]:
    print_run_summary(eval_label, eval_column)


Gebruikte scorecomponenten en ingestelde relatieve gewichten:
  leesprc <- leespercentage / 20, afgerond: 0.15
  inttype <- interactietype (gescoord via eval_type_scores): 0.25
  beoordeling <- beoordeling: 0.75
  leestijd_min <- leestijd_min lineair geschaald naar 0-5: 0.15
Instellingen (1/2): splits=10 | test=30% | train/test=70.0/30.0% | K=3
Instellingen (2/2): buren=5 | overlap=3 | support=1 | drempel=4 | seeds=[11, 21, 31, 41, 51, 61, 71, 81, 91, 1]
Gemiddelde +/- standaardafwijking tussen splits:
Precision@3: 0.0280 +/- 0.0206
NDCG@3: 0.0558 +/- 0.0407
Judged-only Precision@3: 0.1692 +/- 0.1374
Aandeel beoordeeld in aanbevelingen: 0.0440 +/- 0.0126
Studenten met relevante testitems: 24.0000 +/- 4.0000
Studenten met beoordeelde aanbevelingen: 12.4000 +/- 3.5833


In [3]:
# Voer eerst cel 2 uit; die berekent MAE/RMSE per random split.
# MAE is makkelijk te interpreteren; RMSE bestraft grote fouten extra zwaar.
print('Scorevoorspelling op testparen (gemiddelde +/- SD over random splits):')
print_run_summary('MAE', 'MAE')
print_run_summary('RMSE', 'RMSE')
print_run_summary('Voorspellingsdekking (fractie testparen)', 'score_prediction_coverage')

Scorevoorspelling op testparen (gemiddelde +/- SD over random splits):
MAE: 0.9026 +/- 0.0795
RMSE: 1.1421 +/- 0.0864
Voorspellingsdekking (fractie testparen): 0.1973 +/- 0.0087


In [4]:
# Run cel 2 eerst. De baseline voorspelt voor elk testpaar het gemiddelde van de train-scores.
# De vergelijking gebruikt alleen testparen waarvoor collaborative filtering een score kon voorspellen.
eval_comparison = pd.DataFrame({
    'Model': ['Collaborative filtering', 'Gemiddelde train-score baseline'],
    'MAE gemiddeld': [
        eval_run_metrics['MAE'].mean(),
        eval_run_metrics['Global-mean baseline MAE'].mean(),
    ],
    'MAE SD': [
        eval_run_metrics['MAE'].std(ddof=0),
        eval_run_metrics['Global-mean baseline MAE'].std(ddof=0),
    ],
    'RMSE gemiddeld': [
        eval_run_metrics['RMSE'].mean(),
        eval_run_metrics['Global-mean baseline RMSE'].mean(),
    ],
    'RMSE SD': [
        eval_run_metrics['RMSE'].std(ddof=0),
        eval_run_metrics['Global-mean baseline RMSE'].std(ddof=0),
    ],
})
print('Fout op dezelfde voorspelbare testparen (lager is beter):')
print(eval_comparison.round(4).to_string(index=False))
print('Dekking collaborative filtering: {:.1f}%'.format(
    100 * eval_run_metrics['score_prediction_coverage'].mean()
))

Fout op dezelfde voorspelbare testparen (lager is beter):
                          Model  MAE gemiddeld  MAE SD  RMSE gemiddeld  RMSE SD
        Collaborative filtering         0.9026  0.0795          1.1421   0.0864
Gemiddelde train-score baseline         0.7624  0.0367          0.9404   0.0339
Dekking collaborative filtering: 19.7%


In [ ]:
# Fout op dezelfde voorspelbare testparen (lager is beter):
#                           Model  MAE gemiddeld  MAE SD  RMSE gemiddeld  RMSE SD
#         Collaborative filtering         0.9179  0.0487          1.1763   0.0579
# Gemiddelde train-score baseline         0.8061  0.0496          0.9848   0.0465
# Dekking collaborative filtering: 28.6%